In [ ]:
import random
import numpy as np
import pandas as pd
import yfinance as yf
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score, roc_auc_score
from scipy.stats import spearmanr

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

FEATURE_COLS = ['log_ret', 'rv_20', 'vol_z', 'mom_5']
RV_COL_IDX = FEATURE_COLS.index('rv_20') 


def prepare_data(ticker: str, start: str = '2016-01-01', seq_len: int = 30,
                  split_frac: float = 0.8, val_frac: float = 0.15):
    """
    Chronological three-way split: train -> val -> test.
    Target: abs(log_ret), i.e. next-day move SIZE, not direction.
    """
    df = yf.download(ticker, start=start, auto_adjust=True, progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df['log_ret'] = np.log(df['Close'] / df['Close'].shift(1))
    df['rv_20'] = df['log_ret'].rolling(20).std()
    df['vol_z'] = ((df['Volume'] - df['Volume'].rolling(20).mean())
                   / df['Volume'].rolling(20).std())
    df['mom_5'] = df['Close'].pct_change(5)
    df = df.dropna()

    raw_features = df[FEATURE_COLS].values  
    raw_target = np.abs(df['log_ret'].values)

    n = len(raw_features)
    train_end = int(split_frac * n)
    val_end = int((split_frac + (1 - split_frac) * val_frac) * n)

    feature_scaler = StandardScaler().fit(raw_features[:train_end])
    target_scaler = StandardScaler().fit(raw_target[:train_end].reshape(-1, 1))

    scaled_features = feature_scaler.transform(raw_features)
    scaled_target = target_scaler.transform(raw_target.reshape(-1, 1)).flatten()

    def make_windows(feat_arr, tgt_arr, seq):
        X = np.stack([feat_arr[i:i + seq] for i in range(len(feat_arr) - seq)])
        y = np.array([tgt_arr[i + seq] for i in range(len(feat_arr) - seq)])
        return X, y

    Xtr_np, ytr_np = make_windows(scaled_features[:train_end], scaled_target[:train_end], seq_len)
    Xval_np, yval_np = make_windows(scaled_features[train_end:val_end], scaled_target[train_end:val_end], seq_len)
    Xte_np, yte_np = make_windows(scaled_features[val_end:], scaled_target[val_end:], seq_len)

    to_tensor = lambda a: torch.tensor(a, dtype=torch.float32).to(device)
    Xtr, ytr = to_tensor(Xtr_np), to_tensor(ytr_np).unsqueeze(-1)
    Xval, yval = to_tensor(Xval_np), to_tensor(yval_np).unsqueeze(-1)
    Xte, yte = to_tensor(Xte_np), to_tensor(yte_np).unsqueeze(-1)

    def persistence_baseline(offset: int, n_windows: int) -> np.ndarray:
        """
        Predicts tomorrow's |return| as the rv_20 value as of the LAST
        input day of each window -- i.e. "volatility tomorrow = volatility
        as we last knew it," with no lookahead (that day's rv_20 only uses
        data through that day, which the model has already seen as input).
        """
        idx = np.arange(offset + seq_len - 1, offset + seq_len - 1 + n_windows)
        return raw_features[idx, RV_COL_IDX]

    baseline_tr = persistence_baseline(0, len(Xtr_np))
    baseline_val = persistence_baseline(train_end, len(Xval_np))
    baseline_te = persistence_baseline(val_end, len(Xte_np))

    return {
        'Xtr': Xtr, 'ytr': ytr, 'Xval': Xval, 'yval': yval, 'Xte': Xte, 'yte': yte,
        'Xtr_np': Xtr_np, 'Xte_np': Xte_np,
        'baseline_tr': baseline_tr, 'baseline_val': baseline_val, 'baseline_te': baseline_te,
        'feature_scaler': feature_scaler, 'target_scaler': target_scaler,
        'df': df, 'seq_len': seq_len,
    }


class PredictionModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, output_dim):
        super().__init__()
        self.hidden_dim, self.num_layers = hidden_dim, num_layers
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim, device=x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim, device=x.device)
        out, _ = self.lstm(x, (h0, c0))
        return self.fc(out[:, -1, :])


def train_model(Xtr, ytr, Xval=None, yval=None, epochs=200, hidden_dim=32, num_layers=2,
                 lr=1e-3, print_every=25, patience=20):
    input_dim = Xtr.shape[-1]   
    model = PredictionModel(input_dim, hidden_dim, num_layers, 1).to(device)
    criterion = nn.MSELoss()
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    best_val_loss = float('inf')
    best_state = None
    epochs_no_improve = 0

    for epoch in range(epochs):
        model.train()
        pred = model(Xtr)
        loss = criterion(pred, ytr)
        opt.zero_grad()
        loss.backward()
        opt.step()

        val_loss = None
        if Xval is not None:
            model.eval()
            with torch.no_grad():
                val_loss = criterion(model(Xval), yval).item()
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1

        if epoch % print_every == 0:
            msg = f"  epoch {epoch:4d}  train_loss {loss.item():.6f}"
            if val_loss is not None:
                msg += f"  val_loss {val_loss:.6f}  (best {best_val_loss:.6f}, no_improve {epochs_no_improve})"
            print(msg)

        if Xval is not None and epochs_no_improve >= patience:
            if epoch % print_every != 0:
                print(f"  Early stopping at epoch {epoch} (no val improvement for {patience} epochs)")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model


def train_linear_baseline(Xtr_np: np.ndarray, ytr_np: np.ndarray) -> LinearRegression:
    """Same inputs as the LSTM (flattened), simplest possible model."""
    n_samples = Xtr_np.shape[0]
    lr = LinearRegression()
    lr.fit(Xtr_np.reshape(n_samples, -1), ytr_np)
    return lr


def pct_vs_baseline(model_val: float, baseline_val: float, lower_is_better: bool):
    if baseline_val == 0:
        return None, "n/a (baseline is 0)"
    if lower_is_better:
        pct = (baseline_val - model_val) / abs(baseline_val) * 100
    else:
        pct = (model_val - baseline_val) / abs(baseline_val) * 100
    return pct, f"{pct:+.1f}%"

METRIC_DIRECTION = {'rmse': True, 'mae': True, 'r2': False}


def regime_discrimination(pred: np.ndarray, true: np.ndarray, quantile: float = 0.75) -> dict:
    """
    Does predicted magnitude correctly RANK high-vol days above low-vol
    days? AUC treats "is this a top-quartile realized-vol day" as the
    label and predicted magnitude as the score (0.5 = no discrimination,
    1.0 = perfect ranking). Spearman is a complementary, threshold-free
    rank-correlation check.
    """
    threshold = np.quantile(true, quantile)
    label = (true > threshold).astype(int)
    if label.sum() == 0 or label.sum() == len(label):
        return {'auc': float('nan'), 'spearman': float('nan'), 'note': 'Degenerate label split -- try a different quantile.'}

    auc = roc_auc_score(label, pred)
    rho, _ = spearmanr(pred, true)
    return {'auc': auc, 'spearman': rho}


def evaluate(model, Xte, yte, target_scaler, baseline_te, linear_model=None, Xte_np=None) -> dict:
    model.eval()
    with torch.no_grad():
        pred_scaled = model(Xte).cpu().numpy()
    true_scaled = yte.cpu().numpy()

    pred = target_scaler.inverse_transform(pred_scaled).flatten()
    true = target_scaler.inverse_transform(true_scaled).flatten()

    linear_pred = None
    if linear_model is not None and Xte_np is not None:
        n_samples = Xte_np.shape[0]
        linear_pred_scaled = linear_model.predict(Xte_np.reshape(n_samples, -1)).reshape(-1, 1)
        linear_pred = target_scaler.inverse_transform(linear_pred_scaled).flatten()

    results = {'_pred': pred, '_true': true, '_linear_pred': linear_pred, '_baseline_pred': baseline_te}

    def score(pred_arr, true_arr):
        return {
            'rmse': root_mean_squared_error(true_arr, pred_arr),
            'mae': mean_absolute_error(true_arr, pred_arr),
            'r2': r2_score(true_arr, pred_arr),
        }

    model_scores = score(pred, true)
    persistence_scores = score(baseline_te, true)
    linear_scores = score(linear_pred, true) if linear_pred is not None else None

    for m in ['rmse', 'mae', 'r2']:
        lower_better = METRIC_DIRECTION[m]
        pct_p, label_p = pct_vs_baseline(model_scores[m], persistence_scores[m], lower_better)
        entry = {
            'model': model_scores[m], 'baseline_persistence': persistence_scores[m],
            'pct_vs_persistence': pct_p, 'pct_vs_persistence_label': label_p,
            'beats_persistence': (model_scores[m] < persistence_scores[m]) if lower_better else (model_scores[m] > persistence_scores[m]),
        }
        if linear_scores is not None:
            pct_l, label_l = pct_vs_baseline(model_scores[m], linear_scores[m], lower_better)
            entry.update({
                'baseline_linear': linear_scores[m], 'pct_vs_linear': pct_l, 'pct_vs_linear_label': label_l,
                'beats_linear': (model_scores[m] < linear_scores[m]) if lower_better else (model_scores[m] > linear_scores[m]),
            })
        results[m] = entry

    results['regime_discrimination'] = {
        'model': regime_discrimination(pred, true),
        'persistence': regime_discrimination(baseline_te, true),
        'linear': regime_discrimination(linear_pred, true) if linear_pred is not None else None,
    }

    results['prediction_variance_ratio'] = pred.var() / true.var() if true.var() > 0 else float('nan')

    return results


def print_report(results: dict):
    has_linear = results['rmse'].get('baseline_linear') is not None
    header = f"{'Metric':<22}{'Model':>10}{'vs Persist':>13}{'Beats?':>9}"
    if has_linear:
        header += f"{'vs Linear':>12}{'Beats?':>9}"
    print(f"\n{'='*len(header)}")
    print(header)
    print('-' * len(header))
    for name in ['rmse', 'mae', 'r2']:
        r = results[name]
        row = f"{name:<22}{r['model']:>10.5f}{r['pct_vs_persistence_label']:>13}{('YES' if r['beats_persistence'] else 'NO'):>9}"
        if has_linear:
            row += f"{r['pct_vs_linear_label']:>12}{('YES' if r['beats_linear'] else 'NO'):>9}"
        print(row)
    print('=' * len(header))

    rd = results['regime_discrimination']
    print(f"\nRegime discrimination (does predicted magnitude rank high-vol days above low-vol days?):")
    print(f"  {'':<14}{'AUC':>8}{'Spearman':>12}")
    print(f"  {'Model':<14}{rd['model']['auc']:>8.3f}{rd['model']['spearman']:>12.3f}")
    print(f"  {'Persistence':<14}{rd['persistence']['auc']:>8.3f}{rd['persistence']['spearman']:>12.3f}")
    if rd['linear'] is not None:
        print(f"  {'Linear':<14}{rd['linear']['auc']:>8.3f}{rd['linear']['spearman']:>12.3f}")
    print(f"  (AUC 0.5 = no discrimination, 1.0 = perfect ranking)")

    pvr = results['prediction_variance_ratio']
    print(f"\nPrediction variance / actual variance: {pvr:.3f}")
    if pvr < 0.1:
        print(
            "The model is likely just predicting values close to the mean "
            "volatility rather than capturing real regime shifts. Treat any "
            "RMSE/MAE win alongside this number, not in isolation."
        )

def block_bootstrap_rmse_improvement(pred: np.ndarray, baseline_pred: np.ndarray, true: np.ndarray,
                                      block_size: int = 30, n_boot: int = 1000) -> dict:
    """
    PRIMARY significance check. Resamples CONTIGUOUS BLOCKS of days (not
    single days) with replacement, so the resampled series keeps each
    block's internal autocorrelation intact -- unlike resampling individual
    overlapping windows, which would pretend they're independent when
    they're not. Reports the 95% CI of (persistence_RMSE - model_RMSE):
    if the CI excludes 0, the improvement is unlikely to be noise.
    """
    n = len(true)
    n_blocks = max(1, n // block_size)
    diffs = []
    for _ in range(n_boot):
        starts = np.random.randint(0, max(1, n - block_size), size=n_blocks)
        idx = np.concatenate([np.arange(s, min(s + block_size, n)) for s in starts])
        model_rmse = root_mean_squared_error(true[idx], pred[idx])
        baseline_rmse = root_mean_squared_error(true[idx], baseline_pred[idx])
        diffs.append(baseline_rmse - model_rmse)  

    diffs = np.array(diffs)
    ci_low, ci_high = np.percentile(diffs, [2.5, 97.5])
    return {
        'observed_improvement': root_mean_squared_error(true, baseline_pred) - root_mean_squared_error(true, pred),
        'ci_95': (ci_low, ci_high),
        'significant': ci_low > 0, 
    }


def block_bootstrap_auc(pred: np.ndarray, true: np.ndarray, quantile: float = 0.75,
                         block_size: int = 30, n_boot: int = 1000) -> dict:
    """SECONDARY significance check. 95% CI on AUC - 0.5 (chance-level discrimination)."""
    n = len(true)
    n_blocks = max(1, n // block_size)
    aucs = []
    for _ in range(n_boot):
        starts = np.random.randint(0, max(1, n - block_size), size=n_blocks)
        idx = np.concatenate([np.arange(s, min(s + block_size, n)) for s in starts])
        threshold = np.quantile(true[idx], quantile)
        label = (true[idx] > threshold).astype(int)
        if label.sum() == 0 or label.sum() == len(label):
            continue
        aucs.append(roc_auc_score(label, pred[idx]))

    if len(aucs) < 10:
        return {'ci_95': (float('nan'), float('nan')), 'significant': False,
                'note': 'Too few valid bootstrap samples (degenerate label splits).'}

    aucs = np.array(aucs)
    ci_low, ci_high = np.percentile(aucs, [2.5, 97.5])
    return {'observed_auc': roc_auc_score((true > np.quantile(true, quantile)).astype(int), pred),
            'ci_95': (ci_low, ci_high), 'significant': ci_low > 0.5}


def print_significance_report(rmse_boot: dict, auc_boot: dict):
    print(f"\n{'='*70}")
    print("SIGNIFICANCE TESTS (block bootstrap, single run)")
    print('=' * 70)

    print("\nPRIMARY -- RMSE improvement over persistence baseline:")
    print(f"  Observed improvement: {rmse_boot['observed_improvement']:.5f}")
    print(f"  95% CI: [{rmse_boot['ci_95'][0]:.5f}, {rmse_boot['ci_95'][1]:.5f}]  "
          f"({'SIGNIFICANT (CI excludes 0)' if rmse_boot['significant'] else 'not significant (CI includes 0)'})")

    print("\nSECONDARY -- Regime discrimination (AUC vs. chance-level 0.5):")
    if auc_boot.get('note'):
        print(f"  {auc_boot['note']}")
    else:
        print(f"  Observed AUC: {auc_boot['observed_auc']:.3f}")
        print(f"  95% CI: [{auc_boot['ci_95'][0]:.3f}, {auc_boot['ci_95'][1]:.3f}]  "
              f"({'SIGNIFICANT (CI excludes 0.5)' if auc_boot['significant'] else 'not significant (CI includes 0.5)'})")


def multi_seed_evaluation(data: dict, linear_model, n_seeds: int = 10) -> dict:
    """Retrains from scratch with different seeds; reports the DISTRIBUTION
    of RMSE-improvement and AUC, not one run's number."""
    all_rmse_improvement = []
    all_auc = []

    for seed in range(n_seeds):
        torch.manual_seed(seed)
        random.seed(seed)
        np.random.seed(seed)

        model = train_model(data['Xtr'], data['ytr'], Xval=data['Xval'], yval=data['yval'],
                             epochs=500, patience=20, print_every=10_000)
        results = evaluate(model, data['Xte'], data['yte'], data['target_scaler'], data['baseline_te'],
                            linear_model=linear_model, Xte_np=data['Xte_np'])

        rmse_improve = results['rmse']['baseline_persistence'] - results['rmse']['model']
        auc = results['regime_discrimination']['model']['auc']
        all_rmse_improvement.append(rmse_improve)
        all_auc.append(auc)

        print(f"  seed {seed}: rmse_improvement_vs_persistence={rmse_improve:+.5f}, auc={auc:.3f}")

    all_rmse_improvement = np.array(all_rmse_improvement)
    all_auc = np.array(all_auc)

    print(f"\n{'='*70}")
    print(f"MULTI-SEED SUMMARY ({n_seeds} runs)")
    print('=' * 70)
    print(f"RMSE improvement over persistence: mean={all_rmse_improvement.mean():+.5f}, "
          f"std={all_rmse_improvement.std():.5f}, positive in {(all_rmse_improvement > 0).sum()}/{n_seeds} runs")
    print(f"AUC (regime discrimination): mean={all_auc.mean():.3f}, std={all_auc.std():.3f}, "
          f"above 0.5 in {(all_auc > 0.5).sum()}/{n_seeds} runs")

    if (all_rmse_improvement > 0).sum() <= 2 and (all_auc > 0.5).sum() <= 2:
        print("\nImprovement/discrimination shows up in only a couple of seeds -- likely initialization luck, not real skill.")
    elif (all_rmse_improvement > 0).mean() >= 0.7 and (all_auc > 0.5).mean() >= 0.7:
        print("\nConsistent across most seeds -- more credible evidence of real skill.")
    else:
        print("\nMixed across seeds -- no reliable edge established yet.")

    return {'rmse_improvement': all_rmse_improvement, 'auc': all_auc}


if __name__ == "__main__":
    TICKER = "NTES"
    SEQ_LEN = 30

    print(f"Preparing data for {TICKER}")
    data = prepare_data(TICKER, seq_len=SEQ_LEN)
    print(f"Train windows: {len(data['Xtr'])}, Val windows: {len(data['Xval'])}, Test windows: {len(data['Xte'])}")

    print("Training linear-regression baseline (same inputs)")
    linear_model = train_linear_baseline(data['Xtr_np'], data['ytr'].cpu().numpy().flatten())

    print("\n--- Single run (seed unset -- illustrative only, don't trust in isolation) ---")
    model = train_model(data['Xtr'], data['ytr'], Xval=data['Xval'], yval=data['yval'], epochs=500, patience=20)
    results = evaluate(model, data['Xte'], data['yte'], data['target_scaler'], data['baseline_te'],
                        linear_model=linear_model, Xte_np=data['Xte_np'])
    print_report(results)

    pred, true, baseline_pred = results['_pred'], results['_true'], results['_baseline_pred']
    rmse_boot = block_bootstrap_rmse_improvement(pred, baseline_pred, true)
    auc_boot = block_bootstrap_auc(pred, true)
    print_significance_report(rmse_boot, auc_boot)

    print("\n\n--- Multi-seed evaluation (the trustworthy version) ---")
    multi_seed_evaluation(data, linear_model, n_seeds=10)

Preparing data for NTES
Train windows: 2109, Val windows: 50, Test windows: 425
Training linear-regression baseline (same inputs)

--- Single run (seed unset -- illustrative only, don't trust in isolation) ---
  epoch    0  train_loss 0.977477  val_loss 1.324752  (best 1.324752, no_improve 0)
  epoch   25  train_loss 0.940977  val_loss 1.374632  (best 1.299442, no_improve 16)
  Early stopping at epoch 29 (no val improvement for 20 epochs)

Metric                     Model   vs Persist   Beats?   vs Linear   Beats?
---------------------------------------------------------------------------
rmse                     0.01552       +10.0%      YES       +0.9%      YES
mae                      0.01189       +12.3%      YES       -2.7%       NO
r2                      -0.04835       +83.6%      YES      +28.9%      YES

Regime discrimination (does predicted magnitude rank high-vol days above low-vol days?):
                     AUC    Spearman
  Model            0.511       0.055
  Persistenc